# 模型 5 · 冻结 SBERT + TextCNN

**负责人：YU Guanlin**

采用 C3 v1。普通 Jupyter 中直接运行全部单元格即可查看成绩和开发集混淆矩阵。默认查看模式只需 pandas、NumPy、scikit-learn 和 matplotlib。

冻结多语言 SBERT 生成 768 维句向量和最长 82 token 的序列。TextCNN 使用宽度 2、3、4、5 的卷积，池化时屏蔽填充位置，再输出 Neutral、Positive、Negative 三类概率。

实验沿用 `clean_v2` 和 `folds_v1`。每个外折在训练部分用三组内折和三个随机种子比较 C0、C1、C3、C7，再用选出的配置和轮数重训。历史试跑用带早停的耗时确定搜索规模。最终在完整开发集重训的 C3 使用每种卷积核 64 个 filters、学习率 0.0003、dropout 0.5、权重衰减 0.01、9 轮、seed 42，共 689,155 个参数。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score

root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "splits/folds_v1.csv").is_file()),
    None,
)
if root is None:
    raise RuntimeError("请在团队仓库中打开 Notebook")

In [ ]:
history = pd.DataFrame([('selected', 42, 0.6178396072013094, 0.6179262625469312), ('C0', 42, 0.6112929623567921, 0.6099484411225302), ('selected', 17, 0.6153846153846154, 0.6154256967639892), ('C0', 17, 0.602291325695581, 0.6008146310222741), ('selected', 2026, 0.61620294599018, 0.6167064041023497), ('C0', 2026, 0.6104746317512275, 0.6101603675835053)], columns=["Workflow", "Seed", "Accuracy", "Macro-F1"])
summary = history.groupby("Workflow")[["Accuracy", "Macro-F1"]].mean()
summary = summary.rename(index={"selected": "Adopted v1 workflow", "C0": "Matched C0"})
summary.loc["Official TF-IDF + LR"] = [0.5924713584288053, 0.5819958867503482]
display(summary.round(4))

相对同轮 C0，三个种子的 Macro-F1 均有提升，平均提升 **0.0097**。三个按组重采样的 95% 区间分别为 [−0.0095, 0.0252]、[−0.0051, 0.0352]、[−0.0119, 0.0243]。区间均跨零，平均提升低于预设的 0.010。

**0.6167 属于整套选模流程。** 固定 C3 权重用于 303 条保留样本的预测，本实验未用这些样本的标签评分。原始单种子 MVP 中，FastText + TextCNN 和冻结 SBERT + TextCNN 的 Macro-F1 分别为 0.5361 和 0.5796，使用更早的评估流程。

In [ ]:
labels = ["Neutral", "Positive", "Negative"]
run_id = "textcnn_refinement_colab_latest_20261112"
oof = pd.read_csv(root / "Member5_file/evidence/refinement_v1_20261112/selected_seed42_oof.csv")
split = pd.read_csv(root / "splits/folds_v1.csv")
expected_ids = set(split.loc[split["split"] == "train", "id"])
if not oof["id"].is_unique or set(oof["id"]) != expected_ids:
    raise ValueError("OOF 预测与团队开发集 ID 不匹配")

probability_columns = ["prob_0", "prob_1", "prob_2"]
oof_predictions = oof[probability_columns].to_numpy().argmax(axis=1)
print(f"Seed 42 OOF Macro-F1 = {f1_score(oof['label'], oof_predictions, average='macro'):.6f}")
report = classification_report(
    oof["label"], oof_predictions, labels=[0, 1, 2], target_names=labels,
    output_dict=True, zero_division=0,
)
display(pd.DataFrame(report).T.round(4))
ConfusionMatrixDisplay.from_predictions(
    oof["label"], oof_predictions, labels=[0, 1, 2], display_labels=labels, cmap="Blues",
)

## 给组员使用

- 模型汇总读取 `results/m5_textcnn_adopted.csv`。该行为 seed 42 的流程 OOF（Macro-F1 0.617926）；逐条概率位于上方代码读取的 CSV，供错误分析和集成。
- SBERT 句向量及 token 序列可用 `python Member5_file/mvp.py encode --revision 4328cf26390c98c5e3c738b4460a05b95f4911f5 --device cpu` 生成。首次编码需要 transformers 和 PyTorch，可指定自己的 `DATA_ROOT`。
- 以下推理单元格需要原始 C3 权重、配套 token 数组和团队概率包，文件由 5 号在本地保管。默认 `RUN_PREDICTION = False`。开启后还需 PyTorch 和 safetensors。

In [ ]:
RUN_PREDICTION = False

if RUN_PREDICTION:
    import hashlib
    import sys

    from safetensors.torch import load_file

    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    from Member5_file.mvp import TextCNN
    from src.cache import load_embedding
    from src.config import DATA_ROOT
    from src.training import predict_proba

    def checked_file(relative_path, expected_sha256):
        path = DATA_ROOT / relative_path
        if not path.is_file():
            raise FileNotFoundError(f"请先放入 5 号提供的原始文件：{path}")
        with path.open("rb") as handle:
            actual_sha256 = hashlib.file_digest(handle, "sha256").hexdigest()
        if actual_sha256 != expected_sha256:
            raise ValueError(f"原始文件校验不匹配：{path}")
        return path

    cache_hashes = {'tokens_multi.npy': '13f8a548e75aa6404ce49e73510245f753ef4df5f71e8734f5b4d134126c28ab', 'tokens_multi_ids.npy': '7be567b2f824a931bd784ce12af3906905ff8439f9cafdb2d410672f4ce09c5a', 'tokens_mask.npy': '94ebbbf573de58be4579ef1d0e23bf537b0fe9d50c78f3ea77e0272cf60a627c', 'tokens_mask_ids.npy': '7be567b2f824a931bd784ce12af3906905ff8439f9cafdb2d410672f4ce09c5a'}
    for name, digest in cache_hashes.items():
        checked_file(f"emb/{name}", digest)
    weights = checked_file('models/textcnn_c3_v1.safetensors', '417cc498513a9b174b2cc8a76b03a19f9c0129b4938dafb3d74ebb1a7e604a27')
    archive = checked_file('preds/textcnn_refinement_colab_latest_20261112.npz', '38760eb284bc097b702e578d855ca4654da3b2c2e5abb4002083ce889c3bed0d')
    with np.load(archive, allow_pickle=False) as saved:
        if str(saved["run_id"]) != run_id or list(saved["classes"]) != [0, 1, 2]:
            raise ValueError("概率包版本或类别顺序不匹配")
        test_ids = saved["test_ids"]
        expected_proba = saved["test_proba"]
    model = TextCNN(768, filters=64, dropout=0.5)
    model.load_state_dict(load_file(str(weights), device="cpu"))
    probabilities = predict_proba(
        model,
        load_embedding("tokens_multi", ids=test_ids),
        mask=load_embedding("tokens_mask", ids=test_ids),
        batch_size=64,
    )
    np.testing.assert_allclose(probabilities, expected_proba, atol=5e-5, rtol=5e-4)
    np.testing.assert_array_equal(probabilities.argmax(axis=1), expected_proba.argmax(axis=1))
    display(pd.DataFrame(probabilities, columns=labels).assign(id=test_ids))

## Optional training

Install `requirements.txt` and prepare the token and mask cache with `12_emb_tokens.ipynb`.

- `fixed` trains the adopted C3 architecture for nine epochs at seed 42 and writes weights and probabilities for the reserved split.
- `nested` reruns the four candidates on the team folds. Each outer fold keeps its validation groups outside model selection. The matched C0 uses its own selected epoch count. This mode performs 247 fits and can select a different final configuration. It omits the historical timing pilot.
- `off` keeps the existing results review available without re-training.

The replay gives each run a new local folder and neither scores reserved labels nor changes the checked-in result files.

In [ ]:
TRAIN_MODE = "off"   # "fixed" or "nested" to run training
TRAIN_DEVICE = "cpu"  # "mps" or "cuda" if available

if TRAIN_MODE != "off":
    import sys

    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    from Member5_file.c3_workflow import run_c3

    print("New run:", run_c3(mode=TRAIN_MODE, device=TRAIN_DEVICE))